In [ ]:
# MWE that runs Qubitized QPE for a given Hamiltonian

from qarp.operators import QubitOperator
from qarp.operators.functions import count_qubits, eigenspectrum
import numpy as np
from qarp.algorithms import QPE, find_occupation_numbers, dirichlet_kernel_squared


qham = QubitOperator("Z0", -0.2) + QubitOperator("Z1", 0.3) + QubitOperator("X0 X1", 0.4) + QubitOperator("Y0 Y1", -0.5)
#qham = QubitOperator("Z0", -0.4) + QubitOperator("Z1", 0.3) + QubitOperator("Z0 Z1", 0.4)


eigs = eigenspectrum(qham)
print(np.real(qham.sparse_matrix().toarray()))
print("Real eigenvalues: ", eigs)


eigs = eigenspectrum(qham)

# show the eigenvalues and relative eigenstate components to prepare a suitable initial state
n_qubits = count_qubits(qham)
find_occupation_numbers(qham, n_qubits, verbose=True)

In [ ]:
# Build the qubitization circuit with primitives 
from qarp.blocks import QubitizationBlock
from qarp.plotting import plot


qb = QubitizationBlock(qham, operator_name="Scaled hamiltonian").build()
lambda_ = qb.lambda_factor
total_qubitization_qubits = qb.n_qubits
unitaries_qubits = qb.unitaries_qubits

qb.plot()


In [ ]:
# Basis state block 
from qarp.blocks import ComputationalBasisStateBlock
from qarp.blocks import CompositeBlock

state = [1,1]
basis_state = ComputationalBasisStateBlock(state, target_qubits=unitaries_qubits).build()
cs_block = CompositeBlock([basis_state], total_qubitization_qubits).build()


In [ ]:
# Run QPE

n_ancilla = 6
qpe = QPE(cs_block, qb, n_ancilla)

# Build the circuit
qpe.build()

# Run the QPE sampling
qpe.run()

# Access the QPE result
res = qpe.result

# Access the QPE result probability
prob = qpe.result_probability

# The phase should be close to because we probed the |11> state
print(f"The phase is {res:.4f} with probability {prob:.4f}")

In [ ]:
# Plot the phase histogram

fig, ax = qpe.plot(return_fig=True, figsize=(20, 5))
#ax.plot(qpe.freqs, dirichlet_kernel_squared(qpe.freqs, res, 2**n_ancilla), 'r', label="Dirichlet kernel squared")

ax.set_title("QPE with Qubitization (reflection symmetry around x = 0.5)")

In [ ]:
# Get the final energy as cos(2 pi phase). Since we use BlockEncoding, we need to multiply the final result by lambda 

print("Energy of the corresponding found phase: ", lambda_ * np.cos( 2. * np.pi * res  ) )